# CPU scaling of the production one-arm WDM shift

This notebook diagnoses whether the current production kernel is limited by
available CPU parallelism or whether its performance saturates because of
memory/cache pressure.

The production configuration tested here is

```python
assembly_backend = "lagfirst_chunked_lagblock"
assembly_precision = "complex64"
row_chunk_size = 32640
lag_block_size = 17
lag_truncation = 25
```

The important design choice is that every core-count measurement is performed
in a **fresh Python subprocess**. CPU affinity and thread environment variables
are set before JAX/XLA is imported in that subprocess.

The notebook records both wall time and process CPU time. The ratio

\[
N_{\mathrm{eff}} =
\frac{t_{\mathrm{CPU}}}{t_{\mathrm{wall}}}
\]

is an estimate of the average number of CPU cores actively used during the
timed kernel call.

A large `N_eff` with poor additional speedup suggests bandwidth/cache
saturation. A small `N_eff` compared with the allowed core count suggests
underutilisation or insufficient parallel work.

In [ ]:
# --- 1. Notebook configuration ---

from pathlib import Path

PROJECT_ROOT = Path.cwd()

CACHE_DIR = PROJECT_ROOT / ".benchmark_cache" / "one_arm_cpu_scaling"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CACHE = CACHE_DIR / "full_year_wdm_inputs.npz"
REFERENCE_OUTPUT = CACHE_DIR / "reference_output_32640_17.npy"
WORKER_SCRIPT = CACHE_DIR / "one_arm_cpu_scaling_worker.py"
RESULTS_DIR = CACHE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Rebuild only when the waveform/WDM configuration changes.
REBUILD_INPUT_CACHE = False

# Full-year scientific configuration.
N = 3_200_000
dt = 10
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25

alpha = 0.005
tukey_factor = 4

ORBIT_FILE = Path("esa-trailing-orbits.h5")

theta = 3.141592653589793 / 2
phi = 0.0

# Current desktop production schedule.
ROW_CHUNK_SIZE = 32640
LAG_BLOCK_SIZE = 17

# Subprocess timing controls.
WARMUPS = 2
REPEATS = 9

# Set to None for automatic physical-core scaling:
# 1, 2, 4, ..., all detected physical cores.
CORE_COUNTS = None

# The first logical CPU of each physical core is used by default.
# This can be replaced manually after inspecting the topology table.
PHYSICAL_CPU_IDS = None

print("Project root:", PROJECT_ROOT)
print("Cache directory:", CACHE_DIR)
print("Orbit file:", ORBIT_FILE.resolve())


## 2. Detect the desktop CPU topology

This cell uses the current process affinity and, on Linux, `lscpu` topology
data. It selects one logical CPU from each physical core so the first sweep
measures physical-core scaling without adding simultaneous multithreading.

In [ ]:
# --- 2. System and CPU-topology detection ---

import csv
import io
import json
import math
import os
import platform
import shutil
import subprocess
import sys

import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None


def current_affinity():
    if hasattr(os, "sched_getaffinity"):
        return sorted(os.sched_getaffinity(0))
    if psutil is not None:
        return sorted(psutil.Process().cpu_affinity())
    return list(range(os.cpu_count() or 1))


def read_lscpu_topology():
    if shutil.which("lscpu") is None:
        return []

    command = [
        "lscpu",
        "-p=CPU,CORE,SOCKET,NODE,ONLINE",
    ]
    completed = subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True,
    )

    rows = []
    for line in completed.stdout.splitlines():
        if not line or line.startswith("#"):
            continue
        cpu, core, socket, node, online = line.split(",")
        rows.append({
            "cpu": int(cpu),
            "core": int(core),
            "socket": int(socket),
            "node": int(node),
            "online": online.strip().upper() == "Y",
        })
    return rows


AFFINITY = current_affinity()
TOPOLOGY_ROWS = read_lscpu_topology()

if TOPOLOGY_ROWS:
    allowed = set(AFFINITY)
    topology_allowed = [
        row for row in TOPOLOGY_ROWS
        if row["online"] and row["cpu"] in allowed
    ]

    representatives = {}
    for row in sorted(
        topology_allowed,
        key=lambda item: (
            item["socket"],
            item["core"],
            item["cpu"],
        ),
    ):
        key = (row["socket"], row["core"])
        representatives.setdefault(key, row["cpu"])

    detected_physical_cpu_ids = list(representatives.values())
else:
    detected_physical_cpu_ids = list(AFFINITY)

if PHYSICAL_CPU_IDS is None:
    PHYSICAL_CPU_IDS = detected_physical_cpu_ids
else:
    PHYSICAL_CPU_IDS = [int(cpu) for cpu in PHYSICAL_CPU_IDS]

if not PHYSICAL_CPU_IDS:
    raise RuntimeError("No usable CPU ids were detected.")

physical_count = len(PHYSICAL_CPU_IDS)
logical_count = len(AFFINITY)

if CORE_COUNTS is None:
    counts = []
    value = 1
    while value < physical_count:
        counts.append(value)
        value *= 2
    counts.append(physical_count)
    CORE_COUNTS = sorted(set(counts))
else:
    CORE_COUNTS = sorted(
        {
            int(count)
            for count in CORE_COUNTS
            if 1 <= int(count) <= physical_count
        }
    )

print("Platform:", platform.platform())
print("Python:", sys.version.split()[0])
print("Current affinity:", AFFINITY)
print("Logical CPUs available:", logical_count)
print("Detected physical cores:", physical_count)
print("Physical-core representatives:", PHYSICAL_CPU_IDS)
print("Scaling counts:", CORE_COUNTS)

if TOPOLOGY_ROWS and pd is not None:
    display(pd.DataFrame(TOPOLOGY_ROWS))


## 3. Prepare a reusable WDM input cache

The expensive waveform generation and the two WDM transforms are performed
only once. The cache contains:

- the padded time grid;
- the plus-polarisation WDM coefficients;
- the cross-polarisation WDM coefficients;
- the WDM construction parameters.

Each fresh subprocess reconstructs the WDM grid and one-arm plan, but the
reported timing includes only the warmed production response call.

In [ ]:
# --- 3. Prepare or reuse the input cache ---

import time

if REBUILD_INPUT_CACHE or not INPUT_CACHE.exists():
    os.environ["JAX_PLATFORMS"] = "cpu"

    import jax
    import jax.numpy as jnp

    from few.utils.constants import YRSID_SI
    from few.waveform import GenerateEMRIWaveform
    from lisaorbits.file import read_orbit_file

    from WDM_LISAresponse.code.lisa_response.waveforms import (
        generate_windowed_padded_waveform,
    )
    from WDM_LISAresponse.code.lisa_response.response_with_plan import (
        _build_wdm_dT_grid,
        _build_wdm_dT_context_from_wdm,
    )

    # EMRI parameters used by the production-like benchmark.
    model_parameters = {
        "m1": 1e6,
        "m2": 10.0,
        "a": 0.9,
        "p0": 8.5,
        "e0": 0.3,
        "x0": 1.0,
        "dist": 1.0,
        "theta": theta,
        "phi": phi,
        "qK": np.pi / 3,
        "phiK": np.pi / 3,
        "Phi_phi0": np.pi / 3,
        "Phi_theta0": 0.0,
        "Phi_r0": np.pi / 3,
        "eps": 1e-2,
    }

    td_gen = GenerateEMRIWaveform(
        "FastKerrEccentricEquatorialFlux",
        sum_kwargs=dict(pad_output=True, odd_len=True),
        return_list=True,
    )

    orbit_data = read_orbit_file(ORBIT_FILE)
    t0 = orbit_data.t0 + 1.0 * YRSID_SI

    model_kwargs = {
        "td_gen": td_gen,
        **model_parameters,
    }

    window_kwargs = {
        "alpha": alpha,
        "tukey_factor": tukey_factor,
    }

    waveform_start = time.perf_counter()
    times_pad, hplus_td, hcross_td, pad_info = (
        generate_windowed_padded_waveform(
            waveform_model="emri",
            t0=t0,
            dt=dt,
            N=N,
            model_kwargs=model_kwargs,
            window_function="tukey",
            window_kwargs=window_kwargs,
            wdm_Nf=Nf,
            return_padding_info=True,
        )
    )
    waveform_seconds = time.perf_counter() - waveform_start

    grid_start = time.perf_counter()
    grid_context = _build_wdm_dT_grid(
        times_pad,
        Nf=Nf,
        d=d_wdm,
        A_frac=A_frac,
        q=q_wdm,
        calc_m0=True,
    )
    grid_seconds = time.perf_counter() - grid_start

    dwt_start = time.perf_counter()
    wave_context = _build_wdm_dT_context_from_wdm(
        times_pad,
        hplus_td,
        hcross_td,
        grid_context["wdm"],
    )

    w_plus = np.asarray(wave_context["w_t_plus"], dtype=np.complex64)
    w_cross = np.asarray(wave_context["w_t_cross"], dtype=np.complex64)
    dwt_seconds = time.perf_counter() - dwt_start

    np.savez(
        INPUT_CACHE,
        times_pad=np.asarray(times_pad),
        w_plus=w_plus,
        w_cross=w_cross,
        Nf=np.asarray(Nf, dtype=np.int64),
        q_wdm=np.asarray(q_wdm, dtype=np.int64),
        d_wdm=np.asarray(d_wdm, dtype=np.int64),
        A_frac=np.asarray(A_frac, dtype=np.float64),
    )

    print("Created:", INPUT_CACHE)
    print("Waveform generation_s:", f"{waveform_seconds:.3f}")
    print("Grid construction_s:", f"{grid_seconds:.3f}")
    print("Two DWTs_s:", f"{dwt_seconds:.3f}")
    print("WDM shape:", w_plus.shape)
    print("Padding information:", pad_info)
else:
    print("Reusing:", INPUT_CACHE)

with np.load(INPUT_CACHE, allow_pickle=False) as cached:
    print("Cached times:", cached["times_pad"].shape, cached["times_pad"].dtype)
    print("Cached plus:", cached["w_plus"].shape, cached["w_plus"].dtype)
    print("Cached cross:", cached["w_cross"].shape, cached["w_cross"].dtype)

print("Cache size_MiB:", INPUT_CACHE.stat().st_size / 1024**2)


## 4. Write the fresh-process benchmark worker

The worker sets CPU affinity before importing JAX, rebuilds the prepared
one-arm plan, compiles and warms the production kernel, and then reports:

- wall-clock timing;
- process CPU timing;
- effective core count;
- numerical agreement with a saved reference output;
- process memory and thread count.

In [ ]:
# --- 4. Materialise the worker script ---

WORKER_SOURCE = 'from __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport platform\nimport time\nfrom pathlib import Path\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(\n        description="Benchmark the prepared materialized one-arm WDM response "\n        "under a restricted CPU affinity."\n    )\n    parser.add_argument("--input", required=True)\n    parser.add_argument("--orbit-file", required=True)\n    parser.add_argument("--cpu-ids", required=True)\n    parser.add_argument("--warmups", type=int, default=2)\n    parser.add_argument("--repeats", type=int, default=9)\n    parser.add_argument("--row-chunk-size", type=int, default=32640)\n    parser.add_argument("--lag-block-size", type=int, default=17)\n    parser.add_argument("--lag-truncation", type=int, default=25)\n    parser.add_argument("--theta", type=float, required=True)\n    parser.add_argument("--phi", type=float, required=True)\n    parser.add_argument("--reference-output")\n    parser.add_argument("--write-reference", action="store_true")\n    parser.add_argument("--json-output", required=True)\n    return parser.parse_args()\n\n\nargs = parse_args()\n\ncpu_ids = tuple(\n    int(token)\n    for token in args.cpu_ids.split(",")\n    if token.strip()\n)\nif not cpu_ids:\n    raise ValueError("At least one CPU id is required.")\n\n# Restrict the process before importing JAX/XLA.\nif hasattr(os, "sched_setaffinity"):\n    os.sched_setaffinity(0, set(cpu_ids))\n\nn_cpus = len(cpu_ids)\n\n# These are secondary guards. CPU affinity remains the primary restriction.\nfor name in (\n    "OMP_NUM_THREADS",\n    "MKL_NUM_THREADS",\n    "OPENBLAS_NUM_THREADS",\n    "NUMEXPR_NUM_THREADS",\n    "TF_NUM_INTRAOP_THREADS",\n):\n    os.environ[name] = str(n_cpus)\n\nos.environ["TF_NUM_INTEROP_THREADS"] = "1"\nos.environ["JAX_PLATFORMS"] = "cpu"\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\n\ntry:\n    import psutil\nexcept ImportError:\n    psutil = None\n\nfrom lisaorbits.file import read_orbit_file\nfrom lisaorbits import InterpolatedOrbits\n\nfrom WDM_LISAresponse.code.lisa_response.response_with_plan import (\n    _build_wdm_dT_grid,\n)\nfrom WDM_LISAresponse.code.lisa_response.one_arm_plan import (\n    OneArmWDMPlan,\n)\n\n\ndef block_result(value):\n    for leaf in jax.tree_util.tree_leaves(value):\n        if hasattr(leaf, "block_until_ready"):\n            leaf.block_until_ready()\n    return value\n\n\ndef summarize(values):\n    values = np.asarray(values, dtype=float)\n    q25, q75 = np.percentile(values, [25.0, 75.0])\n    return {\n        "median": float(np.median(values)),\n        "minimum": float(np.min(values)),\n        "maximum": float(np.max(values)),\n        "mean": float(np.mean(values)),\n        "std": float(np.std(values)),\n        "q25": float(q25),\n        "q75": float(q75),\n        "iqr": float(q75 - q25),\n        "all": [float(x) for x in values],\n    }\n\n\ninput_path = Path(args.input)\nwith np.load(input_path, allow_pickle=False) as cached:\n    times_pad = np.asarray(cached["times_pad"])\n    w_plus = np.asarray(cached["w_plus"])\n    w_cross = np.asarray(cached["w_cross"])\n    Nf = int(cached["Nf"])\n    q_wdm = int(cached["q_wdm"])\n    d_wdm = int(cached["d_wdm"])\n    A_frac = float(cached["A_frac"])\n\norbit_data = read_orbit_file(args.orbit_file)\norbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)\norbits = InterpolatedOrbits(\n    orbit_times,\n    orbit_data.positions,\n    orbit_data.velocities,\n)\n\ngrid_context = _build_wdm_dT_grid(\n    times_pad,\n    Nf=Nf,\n    d=d_wdm,\n    A_frac=A_frac,\n    q=q_wdm,\n    calc_m0=True,\n)\n\nshift_kwargs = {\n    "assembly_backend": "lagfirst_chunked_lagblock",\n    "assembly_precision": "complex64",\n    "tl_tp_mode": "interp",\n    "tl_tp_interp_points": 32,\n    "tl_tp_interp_kind": "linear",\n    "use_jax": True,\n    "row_chunk_size": int(args.row_chunk_size),\n    "lag_block_size": int(args.lag_block_size),\n    "job_block_size": 1,\n}\n\nplan_build_start = time.perf_counter()\nplan = OneArmWDMPlan.build_from_context(\n    grid_context,\n    orbits,\n    float(args.theta),\n    float(args.phi),\n    L_trunc=int(args.lag_truncation),\n    variable_shift_kwargs=shift_kwargs,\n    use_jax=True,\n)\nplan_build_seconds = time.perf_counter() - plan_build_start\n\nw_plus_device = jnp.asarray(w_plus, dtype=jnp.complex64)\nw_cross_device = jnp.asarray(w_cross, dtype=jnp.complex64)\nblock_result((w_plus_device, w_cross_device))\n\n\ndef run_kernel():\n    return plan.apply_batch_device_materialized(\n        w_plus_device,\n        w_cross_device,\n    )\n\n\nfor _ in range(max(0, int(args.warmups))):\n    block_result(run_kernel())\n\nreference_path = (\n    Path(args.reference_output)\n    if args.reference_output\n    else None\n)\n\noutput = block_result(run_kernel())\n\nif args.write_reference:\n    if reference_path is None:\n        raise ValueError("--write-reference requires --reference-output.")\n    reference_path.parent.mkdir(parents=True, exist_ok=True)\n    np.save(reference_path, np.asarray(output))\n\nrel_l2 = 0.0\nmax_abs_difference = 0.0\n\nif reference_path is not None and reference_path.exists():\n    reference = jnp.asarray(\n        np.load(reference_path, mmap_mode="r"),\n        dtype=jnp.complex64,\n    )\n    delta = output - reference\n    max_abs_value = jnp.max(jnp.abs(delta))\n    reference_norm = jnp.linalg.norm(reference)\n    rel_l2_value = jnp.where(\n        reference_norm > 0,\n        jnp.linalg.norm(delta) / reference_norm,\n        jnp.nan,\n    )\n    block_result((max_abs_value, rel_l2_value))\n    max_abs_difference = float(max_abs_value)\n    rel_l2 = float(rel_l2_value)\n\nwall_times = []\nprocess_cpu_times = []\neffective_core_counts = []\n\nfor _ in range(max(1, int(args.repeats))):\n    cpu_start = time.process_time()\n    wall_start = time.perf_counter()\n\n    timed_output = run_kernel()\n    block_result(timed_output)\n\n    wall_elapsed = time.perf_counter() - wall_start\n    cpu_elapsed = time.process_time() - cpu_start\n\n    wall_times.append(float(wall_elapsed))\n    process_cpu_times.append(float(cpu_elapsed))\n    effective_core_counts.append(\n        float(cpu_elapsed / wall_elapsed)\n        if wall_elapsed > 0\n        else float("nan")\n    )\n\noutput_norm = jnp.linalg.norm(timed_output)\noutput_max_abs = jnp.max(jnp.abs(timed_output))\noutput_sum = jnp.sum(timed_output)\nblock_result((output_norm, output_max_abs, output_sum))\n\nprocess = psutil.Process(os.getpid()) if psutil is not None else None\nrss_bytes = process.memory_info().rss if process is not None else None\nnum_threads = process.num_threads() if process is not None else None\n\nresult = {\n    "platform": platform.platform(),\n    "jax_version": jax.__version__,\n    "jax_backend": jax.default_backend(),\n    "jax_devices": [str(device) for device in jax.devices()],\n    "requested_cpu_ids": list(cpu_ids),\n    "effective_affinity": (\n        sorted(os.sched_getaffinity(0))\n        if hasattr(os, "sched_getaffinity")\n        else list(cpu_ids)\n    ),\n    "available_cpu_count": n_cpus,\n    "row_chunk_size": int(args.row_chunk_size),\n    "lag_block_size": int(args.lag_block_size),\n    "lag_truncation": int(args.lag_truncation),\n    "wdm_shape": [int(w_plus.shape[0]), int(w_plus.shape[1])],\n    "output_shape": [int(x) for x in timed_output.shape],\n    "output_dtype": str(timed_output.dtype),\n    "plan_build_seconds": float(plan_build_seconds),\n    "wall_seconds": summarize(wall_times),\n    "process_cpu_seconds": summarize(process_cpu_times),\n    "effective_core_count": summarize(effective_core_counts),\n    "max_abs_difference_from_reference": max_abs_difference,\n    "relative_l2_difference_from_reference": rel_l2,\n    "output_norm": float(output_norm),\n    "output_max_abs": float(output_max_abs),\n    "output_sum_real": float(jnp.real(output_sum)),\n    "output_sum_imag": float(jnp.imag(output_sum)),\n    "rss_bytes": rss_bytes,\n    "process_thread_count_after_run": num_threads,\n}\n\noutput_json = Path(args.json_output)\noutput_json.parent.mkdir(parents=True, exist_ok=True)\noutput_json.write_text(\n    json.dumps(result, indent=2),\n    encoding="utf-8",\n)\n\nprint(json.dumps(result, indent=2))\n'

WORKER_SCRIPT.write_text(WORKER_SOURCE, encoding='utf-8')
print('Worker script:', WORKER_SCRIPT)


## 5. Run the physical-core scaling sweep

The one-core run creates the reference output. Every later subprocess compares
its output against that reference using a relative \(L_2\) difference.

The CPU ids are selected compactly from the detected physical-core
representatives. On a multi-socket system this usually fills the first socket
before moving to the next one, which makes the onset of cross-socket effects
visible.

In [ ]:
# --- 5. Launch one fresh subprocess per core count ---

import json
import subprocess
import sys
from datetime import datetime

if not ORBIT_FILE.exists():
    raise FileNotFoundError(
        f"Orbit file not found: {ORBIT_FILE.resolve()}"
    )

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = RESULTS_DIR / run_stamp
run_dir.mkdir(parents=True, exist_ok=True)

scaling_results = []

for index, core_count in enumerate(CORE_COUNTS):
    selected_cpu_ids = PHYSICAL_CPU_IDS[:core_count]
    output_json = run_dir / f"cores_{core_count:03d}.json"

    command = [
        sys.executable,
        str(WORKER_SCRIPT),
        "--input",
        str(INPUT_CACHE),
        "--orbit-file",
        str(ORBIT_FILE),
        "--cpu-ids",
        ",".join(str(cpu) for cpu in selected_cpu_ids),
        "--warmups",
        str(WARMUPS),
        "--repeats",
        str(REPEATS),
        "--row-chunk-size",
        str(ROW_CHUNK_SIZE),
        "--lag-block-size",
        str(LAG_BLOCK_SIZE),
        "--lag-truncation",
        str(L_TRUNC),
        "--theta",
        repr(float(theta)),
        "--phi",
        repr(float(phi)),
        "--reference-output",
        str(REFERENCE_OUTPUT),
        "--json-output",
        str(output_json),
    ]

    if index == 0 or not REFERENCE_OUTPUT.exists():
        command.append("--write-reference")

    print(
        f"\nRunning {core_count} physical core(s):",
        selected_cpu_ids,
    )

    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )

    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError(
            f"Worker failed for {core_count} core(s)."
        )

    result = json.loads(output_json.read_text(encoding="utf-8"))
    scaling_results.append(result)

    wall = result["wall_seconds"]
    effective = result["effective_core_count"]

    print(
        f"median wall={wall['median']:.4f} s | "
        f"IQR={wall['iqr']:.4f} s | "
        f"effective cores={effective['median']:.2f} | "
        f"rel_l2={result['relative_l2_difference_from_reference']:.3e}"
    )

print("\nResults directory:", run_dir)


## 6. Summarise scaling and CPU utilisation

`effective_core_fraction` compares the measured process CPU time with the
number of cores made available to the subprocess:

\[
f_{\mathrm{eff}} =

rac{N_{\mathrm{eff}}}{N_{\mathrm{allowed}}}.
\]

This is not a hardware-counter measurement, but it is a useful first
diagnostic of whether XLA is keeping the allowed cores busy.

In [ ]:
# --- 6. Build and save the scaling table ---

rows = []

one_core_wall = scaling_results[0]["wall_seconds"]["median"]

for result in scaling_results:
    allowed = int(result["available_cpu_count"])
    wall = result["wall_seconds"]
    effective = result["effective_core_count"]

    speedup = one_core_wall / wall["median"]
    efficiency = speedup / allowed
    effective_fraction = effective["median"] / allowed

    rows.append({
        "allowed_physical_cores": allowed,
        "median_wall_s": wall["median"],
        "q25_wall_s": wall["q25"],
        "q75_wall_s": wall["q75"],
        "iqr_wall_s": wall["iqr"],
        "speedup_vs_1_core": speedup,
        "parallel_efficiency": efficiency,
        "median_effective_cores": effective["median"],
        "effective_core_fraction": effective_fraction,
        "relative_l2": result[
            "relative_l2_difference_from_reference"
        ],
        "rss_gib": (
            result["rss_bytes"] / 1024**3
            if result["rss_bytes"] is not None
            else np.nan
        ),
        "process_threads_after_run": result[
            "process_thread_count_after_run"
        ],
    })

if pd is not None:
    scaling_table = pd.DataFrame(rows)
    display(scaling_table)
    scaling_table.to_csv(
        run_dir / "cpu_scaling_summary.csv",
        index=False,
    )
else:
    scaling_table = rows
    for row in rows:
        print(row)

(run_dir / "cpu_scaling_summary.json").write_text(
    json.dumps(rows, indent=2),
    encoding="utf-8",
)

print("Saved summary in:", run_dir)


## 7. Scaling plots

The ideal-speedup line assumes perfect linear scaling. Deviation from it is
expected; the key question is where the measured curve begins to plateau.

In [ ]:
# --- 7a. Wall time versus allowed physical cores ---

import matplotlib.pyplot as plt

core_values = [row["allowed_physical_cores"] for row in rows]
wall_values = [row["median_wall_s"] for row in rows]

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(core_values, wall_values, marker="o")
ax.set_xlabel("Allowed physical cores")
ax.set_ylabel("Median wall time [s]")
ax.set_title("Production WDM shift: physical-core scaling")
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
# --- 7b. Measured speedup versus ideal speedup ---

speedup_values = [row["speedup_vs_1_core"] for row in rows]

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(core_values, speedup_values, marker="o", label="Measured")
ax.plot(core_values, core_values, linestyle="--", label="Ideal")
ax.set_xlabel("Allowed physical cores")
ax.set_ylabel("Speedup relative to one core")
ax.set_title("Production WDM shift: parallel speedup")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


In [ ]:
# --- 7c. Effective cores used ---

effective_values = [row["median_effective_cores"] for row in rows]

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(core_values, effective_values, marker="o", label="Measured")
ax.plot(core_values, core_values, linestyle="--", label="Allowed")
ax.set_xlabel("Allowed physical cores")
ax.set_ylabel("Median effective cores")
ax.set_title("Process CPU time divided by wall time")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 8. Heuristic interpretation

The following classification is intentionally conservative:

- **Underutilised:** the full-core run uses substantially fewer effective
  cores than were allowed.
- **Likely bandwidth/cache limited:** most cores are active, but speedup has
  already plateaued.
- **Still scaling:** the final step in core count produces a meaningful
  reduction in wall time.

This is a diagnostic guide, not a replacement for hardware counters.

In [ ]:
# --- 8. Print a cautious interpretation ---

last = rows[-1]
first = rows[0]

full_effective_fraction = last["effective_core_fraction"]
full_parallel_efficiency = last["parallel_efficiency"]

if len(rows) >= 2:
    previous = rows[-2]
    final_step_improvement = (
        1.0
        - last["median_wall_s"] / previous["median_wall_s"]
    )
else:
    final_step_improvement = np.nan

print(
    "Full-core effective fraction:",
    f"{100.0 * full_effective_fraction:.1f}%",
)
print(
    "Full-core parallel efficiency:",
    f"{100.0 * full_parallel_efficiency:.1f}%",
)
print(
    "Improvement from penultimate to full core count:",
    (
        f"{100.0 * final_step_improvement:.1f}%"
        if np.isfinite(final_step_improvement)
        else "not available"
    ),
)

if full_effective_fraction < 0.60:
    conclusion = (
        "The kernel appears to underutilise the allowed physical cores. "
        "The next investigation should focus on how XLA exposes row/job "
        "parallelism and on thread-pool behaviour."
    )
elif (
    np.isfinite(final_step_improvement)
    and final_step_improvement < 0.05
):
    conclusion = (
        "Most allowed cores appear active, but wall-time scaling has "
        "largely plateaued. Memory bandwidth, cache pressure, gathers, "
        "or reductions are more likely limits than a shortage of threads."
    )
else:
    conclusion = (
        "The kernel is still obtaining a meaningful benefit from the "
        "largest tested core count. More CPU parallelism may remain useful."
    )

print("\nInterpretation:")
print(conclusion)
print(
    "\nDo not change the production kernel from this heuristic alone. "
    "Use the table to decide whether the next experiment should be "
    "thread/parallelism work or a memory-layout optimisation such as "
    "contiguous Tl/Tp lag consumption."
)


## 9. Outputs to report

Please retain or paste:

1. the detected CPU model/topology and physical-core representatives;
2. the `cpu_scaling_summary.csv` table;
3. the three plots;
4. the heuristic interpretation;
5. any worker failure messages.

The most important columns are:

```text
allowed_physical_cores
median_wall_s
speedup_vs_1_core
parallel_efficiency
median_effective_cores
effective_core_fraction
relative_l2
```

No desktop specifications are required in advance because this notebook
detects them. The detected topology output is still useful for checking that
the automatic physical-core selection was sensible.